# Pedestrian tracking: frame-to-trajectory storyboard

This lesson turns a local video into per-frame pedestrian detections, persistent track IDs, a trajectory table, a heatmap, and an optional rendered clip. The default is a bounded local smoke path. External video acquisition is opt-in, and all generated media stays under the ignored `output/` directory.

![YOLO tracking storyboard](../../../docs/figs/yolo_tracking_storyboard.svg)

Detection and tracking are different: the detector proposes boxes and confidence scores for each frame; ByteTrack associates boxes over time. Occlusion, camera motion, missed detections, and perspective mean that a trajectory is an observation from this camera—not a complete account of pedestrian behaviour.

## Step 1: Testing YOLO v11 of detecting and tracking pedestrain

In [ ]:
from IPython.display import display, clear_output
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO
import os
from ccai9012.paths import STARTER_KITS_DIR

VIDEO_DIR = STARTER_KITS_DIR / "4_cv_models" / "webcam_yolo" / "video"
OUTPUT_DIR = STARTER_KITS_DIR / "4_cv_models" / "webcam_yolo" / "output"
MODEL_PATH = STARTER_KITS_DIR / "4_cv_models" / "webcam_yolo" / "yolo11m.pt"
MODEL_FAMILY = "YOLO11"
MODEL_WEIGHTS = "yolo11m.pt"

In [ ]:
# Load the pinned local YOLO11 medium model; do not silently migrate generations.
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MODEL_PATH}. Download the official {MODEL_FAMILY} "
        f"{MODEL_WEIGHTS} file into the Starter Kit directory, then rerun."
    )
model = YOLO(str(MODEL_PATH))
print(f"Using {MODEL_FAMILY} weights: {MODEL_PATH}")

In [ ]:
def detect_track(video_path, output_dir, max_frames=None, show_window=False):
    # make sure the output_dir exists
    os.makedirs(output_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open local video: {video_path}")
    
    # Get original video info
    fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # VideoWriter will be initialized after first annotated frame (to match exact resolution)
    save_output = True
    out = None
    video_name = os.path.splitext(os.path.basename(str(video_path)))[0]
    output_path = os.path.join(output_dir, f"{video_name}_output.mp4")
    
    # Prepare list for detection results
    all_detections = []
    frame_index = 0
    
    # Start reading video
    while cap.isOpened():
        if max_frames is not None and frame_index >= max_frames:
            break
        ret, frame = cap.read()
        if not ret:
            break
    
        # Perform tracking (ByteTrack), detecting only person class (class 0)
        results = model.track(frame, persist=True, classes=[0], tracker="bytetrack.yaml", verbose=False)
        annotated = results[0].plot()  # Annotated frame for visualization
    
        # Initialize VideoWriter if needed (after first annotated frame)
        if save_output and out is None:
            ann_height, ann_width = annotated.shape[:2]
            out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (ann_width, ann_height))
    
        # Extract bounding boxes and IDs (if any)
        bboxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id
        if ids is not None:
            ids = ids.int().tolist()
        else:
            ids = [-1] * len(bboxes)  # -1 means untracked
    
        # Record detection results to list
        for obj_id, bbox in zip(ids, bboxes):
            x1, y1, x2, y2 = bbox
            all_detections.append({
                "frame": frame_index,
                "id": int(obj_id),
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2)
            })
    
        if show_window:
            img_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            clear_output(wait=True)
            display(Image.fromarray(img_rgb))
    
        # Save current frame to output video
        if save_output:
            out.write(annotated)
    
        frame_index += 1
    
    # Release video and save results
    cap.release()
    if out is not None:
        out.release()
    cv2.destroyAllWindows()
    
    # Save detection results to CSV
    df = pd.DataFrame(all_detections)
    csv_path = os.path.join(output_dir, f"{video_name}_results.csv")
    df.to_csv(csv_path, index=False)
    print(f"Tracking complete: {len(df)} detections saved to {csv_path}")
    return df

In [ ]:
SMOKE_MODE = True
RUN_FULL_VIDEO = False
SMOKE_VIDEO = VIDEO_DIR / "closed_test_15.mp4"
FULL_VIDEO = VIDEO_DIR / "nuclear_test_15.mp4"
video_path = SMOKE_VIDEO if SMOKE_MODE else FULL_VIDEO
if not video_path.exists():
    raise FileNotFoundError(
        f"Missing {video_path}. Add a short local clip under {VIDEO_DIR}; "
        "the notebook does not download video automatically."
    )
df_test = detect_track(video_path, OUTPUT_DIR, max_frames=60 if SMOKE_MODE else None, show_window=False)

In [ ]:
# CSV Output Explanation:
# The CSV file 'yolo_track_results.csv' contains one row per detected object per frame.
# Each row includes the following columns:
#
# - frame:        The index of the video frame where the detection occurred (0-based).
# - id:           The tracking ID assigned by ByteTrack.
#                 Objects with the same ID across frames are considered the same tracked person.
#                 An ID of -1 indicates that no stable tracking ID was assigned (e.g., early frames or low confidence).
# - x1, y1:       The coordinates of the top-left corner of the bounding box (in pixels).
# - x2, y2:       The coordinates of the bottom-right corner of the bounding box (in pixels).
#
# These coordinates are relative to the resolution of the original input frame.

# The schema is one row per detected person per frame.
df_test[["frame", "id", "x1", "y1", "x2", "y2"]].head()

## Step 2: Optional external video acquisition

In [ ]:
RUN_EXTERNAL_CLIP = False
EXTERNAL_VIDEO = VIDEO_DIR / "Live Cam Iseo _ SkylineWebcams _(9_30_16_55).mp4"
if RUN_EXTERNAL_CLIP:
    if not EXTERNAL_VIDEO.exists():
        raise FileNotFoundError(
            f"Missing {EXTERNAL_VIDEO}. Download a permitted clip manually, "
            "record its source/licence and place it under video/."
        )
    from moviepy import VideoFileClip
    clip = VideoFileClip(str(EXTERNAL_VIDEO)).subclipped(0, 60)
    clip.write_videofile(str(VIDEO_DIR / "cam_iseo.mp4"), fps=1, codec="mpeg4", audio=False)
else:
    print("External acquisition disabled; use the local smoke clip.")

In [ ]:
# The optional acquisition cell above creates a short local clip only when enabled.
if RUN_EXTERNAL_CLIP:
    print(f"Created {VIDEO_DIR / 'cam_iseo.mp4'}")

## Step 3: Run YOLO to detect and track pedestrian in the clip and save the results

In [ ]:
if RUN_FULL_VIDEO:
    if not FULL_VIDEO.exists():
        raise FileNotFoundError(f"Missing {FULL_VIDEO}; add a local permitted clip first.")
    detect_track(FULL_VIDEO, OUTPUT_DIR, show_window=False)
else:
    print("Full-video run disabled; the bounded smoke result is sufficient for the first pass.")

## Step 4: Optional table-to-trajectory visualisation

In [ ]:
RUN_VISUALIZATION = False
analysis_video = SMOKE_VIDEO if SMOKE_MODE else FULL_VIDEO
analysis_csv = OUTPUT_DIR / f"{analysis_video.stem}_results.csv"
if not analysis_csv.exists():
    print(f"No tracking table at {analysis_csv}; run detection first.")
else:
    df = pd.read_csv(analysis_csv)

    # Calculate bottom-centre points; these are image coordinates, not map coordinates.
    df['x_center'] = (df['x1'] + df['x2']) / 2
    df['y_bottom'] = df['y2']

In [ ]:
# save a frame from the video as background
video_path = str(analysis_video)
frame_number = 0
background_img_path = OUTPUT_DIR / "viz" / "bg_frame.jpg"
background_img_path.parent.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
ret, frame = cap.read()
cap.release()

if ret:
    cv2.imwrite(str(background_img_path), frame)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import seaborn as sns

# Load the tracking table produced by the selected local video.
df = pd.read_csv(analysis_csv)
df['x_center'] = (df['x1'] + df['x2']) / 2
df['y_bottom'] = df['y2']

# Load background image
bg_img = mpimg.imread(str(background_img_path))
height, width = bg_img.shape[:2]

In [ ]:
# Visualize pedestrian footprint heatmap

plt.figure(figsize=(12, 9))

# Show background image with transparency
plt.imshow(bg_img, extent=[0, width, height, 0], alpha=0.9)

# KDE heatmap with smooth blending and no white contour lines
sns.kdeplot(
    x=df['x_center'],
    y=df['y_bottom'],
    fill=True,              # fill area under KDE
    cmap="coolwarm",            # smooth color map
    bw_adjust=1.0,          # bandwidth
    levels=100,             # more levels = smoother gradient
    alpha=0.3,              # transparency of the heatmap
    thresh=0.02,            # remove low-density noise
    linewidths=0            # no contour lines (white borders)
)

plt.title("Pedestrian Density Heatmap (Smooth Overlay)")
plt.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "viz" / "heatmap_overlay.png", dpi=300)
plt.show()

In [ ]:
# Pedestrian Trajectories
plt.figure(figsize=(12, 9))

# Show background image
plt.imshow(bg_img, extent=[0, width, height, 0], alpha=0.5)

# Draw each trajectory
for pid, group in df.groupby("id"):
    plt.plot(group['x_center'], group['y_bottom'], linewidth=1, alpha=0.5)

plt.title("Pedestrian Trajectories")
plt.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "viz" / "trajectory_overlay.png", dpi=300)
plt.show()

## Step 5: Optional rendered trajectory clip

In [ ]:
# Rendering is optional because it writes a video; keep generated media local and ignored.
RUN_RENDERED_CLIP = False
if RUN_RENDERED_CLIP:
    from visualize_trajectories import visualize_video

In [ ]:
if RUN_RENDERED_CLIP:
    visualize_video(
        input_csv=str(analysis_csv),
        input_video=str(analysis_video),
        output_video=str(OUTPUT_DIR / "viz" / f"{analysis_video.stem}_viz.mp4"),
        show_window=False
    )
else:
    print("Rendered clip disabled; frame table and still visualisations are sufficient for smoke QA.")